In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install groq neo4j joblib nltk

import re
import joblib
import nltk
from nltk.corpus import stopwords
from neo4j import GraphDatabase
from groq import Groq
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 13.4 MB/s eta 0:00:00


In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
model_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/intent_model.pkl"
vectorizer_path = "/content/drive/MyDrive/banking-ai-knowledge-system/nlp_model/vectorizer.pkl"

model = joblib.load(model_path)
vectorizer = joblib.load(vectorizer_path)

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [ ]:
def normalize_intent(intent):
    intent = intent.lower()

    if "lost" in intent or "stolen" in intent:
        return "LostCard"

    if "activate" in intent:
        return "ActivateCard"

    if "close" in intent:
        return "CloseAccount"

    return None

In [ ]:
uri = "neo4j+s://YOUR_DB_ID.databases.neo4j.io"
username = "neo4j"
password = "YOUR_PASSWORD"

In [ ]:
def get_graph_response(intent_name):

    query = """
    MATCH (i:Intent {name:$intent})-[:TRIGGERS]->(s)-[:REQUIRES]->(v)
    RETURN s.name AS service, v.name AS verification
    """

    with driver.session() as session:
        result = session.run(query, intent=intent_name)
        return [record.data() for record in result]

In [ ]:
def build_context(result):

    if not result:
        return "No information available."

    service = result[0]['service']
    verification = result[0]['verification']

    return f"""
    Service: {service}
    Verification: {verification}
    """

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

In [ ]:
def rag_response(user_query):

    cleaned = clean_text(user_query)
    vec = vectorizer.transform([cleaned])
    predicted_intent = model.predict(vec)[0]

    graph_intent = normalize_intent(predicted_intent)
    graph_result = get_graph_response(graph_intent)

    context = build_context(graph_result)

    prompt = f"""
    You are a banking assistant.

    User query:
    {user_query}

    Knowledge:
    {context}

    Provide a helpful response.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
rag_response("My card was stolen")

"I'm so sorry to hear that your card was stolen. Don't worry, I'm here to help. To ensure your security, I'll need to verify your identity before we proceed. This is a standard process to protect your account.\n\nCan you please confirm your identity with me? I'll ask you a few questions to verify your information. Once your identity is verified, I can assist you with replacing your stolen card through our CardReplacement service.\n\nPlease let me know if you're ready to proceed with the identity verification process."